# HEXA UDON — MAPPO Cloud Training Notebook

Notebook này được thiết kế để chạy huấn luyện mô hình Multi-Agent PPO (MAPPO) cho bài toán HEXA UDON trên môi trường Google Colab sử dụng GPU (T4/V100/A100).

## Hướng dẫn chuẩn bị trước khi chạy:
1. Nén toàn bộ thư mục dự án của bạn dưới dạng file `.zip` (tên là `procon2026.zip`).
2. Kéo thả file `procon2026.zip` vào bảng điều khiển Files ở cạnh trái Colab (hoặc upload qua mã code phía dưới).
3. Bật GPU: **Runtime -> Change runtime type -> T4 GPU** (hoặc GPU bất kỳ bạn có) -> **Save**.

In [ ]:
# 1. Kiểm tra thông tin GPU
!nvidia-smi

## Kết nối Google Drive
Kết nối Google Drive để lưu các checkpoint (`model.pt`) tự động sau mỗi 500 episodes. Điều này đảm bảo bạn không bị mất dữ liệu huấn luyện khi Google Colab reset runtime do hết thời hạn (12 tiếng) hoặc mất kết nối.

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục lưu trữ checkpoints trên Drive
!mkdir -p /content/drive/MyDrive/procon2026

## Giải nén mã nguồn dự án

In [ ]:
# 3. Giải nén code dự án
# Lưu ý: Đảm bảo bạn đã upload file 'procon2026.zip' lên thư mục /content/ trước khi chạy cell này
import os

zip_path = "/content/procon2026.zip"
if os.path.exists(zip_path):
    # Xóa thư mục cũ nếu có trước khi giải nén để tránh xung đột
    !rm -rf /content/procon2026
    !unzip -q {zip_path} -d /content/procon2026
    print("Giải nén thành công!")
else:
    print("LỖI: Chưa tìm thấy file procon2026.zip ở thư mục /content/. Hãy upload file zip lên trước.")

## Cài đặt Thư viện và Kiểm tra Môi trường

In [ ]:
# 4. Di chuyển vào thư mục chứa code và cài đặt các thư viện bổ sung
import os

# Tự động phát hiện cấu trúc thư mục lồng nhau (do zip)
if os.path.exists("/content/procon2026/procon2026"):
    %cd /content/procon2026/procon2026
else:
    %cd /content/procon2026

!pip install requests tensorboard -q

# Kiểm tra PyTorch có nhận GPU không
import torch
print(f"Thư mục hiện tại: {os.getcwd()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## Khởi động TensorBoard theo dõi quá trình train
TensorBoard cho phép bạn xem biểu đồ Loss, Reward, số lượng Series thu thập được và cấp độ Curriculum theo thời gian thực.

In [ ]:
# 5. Chạy TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/runs/mappo

## LỰA CHỌN A: Huấn luyện mới từ đầu (Train from scratch)
Chỉ chạy cell dưới đây nếu bạn muốn huấn luyện mô hình mới hoàn toàn và ghi đè lên file checkpoint cũ trên Drive.

In [ ]:
# 6. Tiến hành huấn luyện mới
import os

if os.path.exists("/content/procon2026/procon2026"):
    %cd /content/procon2026/procon2026
else:
    %cd /content/procon2026

model_dir = "/content/drive/MyDrive/procon2026"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "model.pt")

print("[Colab] Đang khởi chạy huấn luyện mới từ đầu...")

!python src/main.py train \
  --episodes 20000 \
  --curriculum \
  --selfplay \
  --device cuda \
  --seed 42 \
  --log-every 100 \
  --save-every 500 \
  --log-dir /content/runs/mappo \
  --save {model_path}

## LỰA CHỌN B: Tải lại model từ Google Drive và huấn luyện tiếp (Resume training)
Chỉ chạy cell dưới đây nếu bạn bị ngắt kết nối Colab giữa chừng hoặc muốn huấn luyện tiếp tục dựa trên file `model.pt` có sẵn trong Google Drive.

In [ ]:
# 7. Load model và huấn luyện tiếp
import os

if os.path.exists("/content/procon2026/procon2026"):
    %cd /content/procon2026/procon2026
else:
    %cd /content/procon2026

model_dir = "/content/drive/MyDrive/procon2026"
model_path = os.path.join(model_dir, "model.pt")

if os.path.exists(model_path):
    print(f"[Colab] Tìm thấy checkpoint cũ tại {model_path}. Tiến hành resume training...")
    
    !python src/main.py train \
      --episodes 20000 \
      --curriculum \
      --selfplay \
      --device cuda \
      --seed 42 \
      --log-every 100 \
      --save-every 500 \
      --log-dir /content/runs/mappo \
      --load {model_path} \
      --save {model_path}
else:
    print(f"LỖI: Không tìm thấy file checkpoint tại {model_path} để tải. Vui lòng chạy Lựa chọn A trước.")